### **[Hybrid Machine Learning for Market Regime Detection](https://medium.com/@alexzap922/9b0c7a0bf0f2)**

> *Part 2: Second-Stage Validation & Revision of Market Regime Detection via Machine Learning Clustering & Classification in Python*

In [ ]:
import os
import sys

import warnings
warnings.filterwarnings("ignore")

import requests

import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import yellowbrick

import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import silhouette_score, accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier

from eodhd import APIClient
import mplfinance as mpf

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  API_KEY = userdata.get('EODHD_API_KEY')
else:
  API_KEY = os.getenv('EODHD_API_KEY')

%autosave 20

In [ ]:
client = APIClient(API_KEY)
start_date = "2020-01-02"
end_date = "2026-05-23" # the actual date is 2026-05-22

def fetch_eod(symbol, start_date, end_date):
  """
  Fetch EOD data for a given symbol from EODHD.
  """
  url = f"https://eodhd.com/api/eod/{symbol}"
  params = {
    "api_token": API_KEY,
    "from": start_date,
    "to": end_date,
    "fmt": "json"
  }
  resp = requests.get(url, params=params)
  data = resp.json()
  df = pd.DataFrame(data)
  df = df.sort_values("date")
  df["date"] = pd.to_datetime(df["date"])
  df.set_index("date", inplace=True)
  return df


#### **VTI**

In [ ]:
# Fetch EOD data
spy_df = fetch_eod("VTI.US", start_date, end_date)
display(spy_df.info())

df = spy_df.copy()
# --- Bollinger Bands (20-period normally; here small dataset so just demo) ---
window = 20  # use 20 in real data
df["MA"] = df["close"].rolling(window).mean()
df["STD"] = df["close"].rolling(window).std()

df["Upper"] = df["MA"] + 2 * df["STD"]
df["Lower"] = df["MA"] - 2 * df["STD"]

# --- Define additional plots for mplfinance ---
add_plots = [
  mpf.make_addplot(df["MA"], color="blue",alpha=0.6),
  mpf.make_addplot(df["Upper"], color="green",alpha=0.4),
  mpf.make_addplot(df["Lower"], color="red",alpha=0.4),
]

# --- Plot ---
mpf.plot(
  df,
  type="candle",
  volume=True,
  style="charles",
  addplot=add_plots,
  title="VTI.US Candlesticks with Bollinger Bands & Volume",
  ylabel="Price USD",figsize=(14, 6), ylim=(100, 370),
  ylabel_lower="Volume",xlabel="Date"
)

#### **IWO**

In [ ]:
iwm_df = fetch_eod("IWO.US", start_date, end_date)

# Visualization
df = pd.DataFrame()
df = iwm_df.copy()

# --- Bollinger Bands (20-period normally; here small dataset so just demo) ---
window = 20  # use 20 in real data
df["MA"] = df["close"].rolling(window).mean()
df["STD"] = df["close"].rolling(window).std()

df["Upper"] = df["MA"] + 2 * df["STD"]
df["Lower"] = df["MA"] - 2 * df["STD"]

# --- Define additional plots for mplfinance ---
add_plots = [
  mpf.make_addplot(df["MA"], color="blue",alpha=0.6),
  mpf.make_addplot(df["Upper"], color="green",alpha=0.4),
  mpf.make_addplot(df["Lower"], color="red",alpha=0.4),
]

# --- Plot ---
mpf.plot(
  df,
  type="candle",
  volume=True,
  style="charles",
  addplot=add_plots,
  title="IWO.US Candlesticks with Bollinger Bands & Volume",
  ylabel="Price USD",figsize=(14, 6), ylim=(100, 370),
  ylabel_lower="Volume",xlabel="Date"
)

#### **JNK**

In [ ]:
hyg_df = fetch_eod("JNK", start_date, end_date)

# Visualization
df = hyg_df.copy()

# --- Bollinger Bands (20-period normally; here small dataset so just demo) ---
window = 20  # use 20 in real data
df["MA"] = df["close"].rolling(window).mean()
df["STD"] = df["close"].rolling(window).std()

df["Upper"] = df["MA"] + 2 * df["STD"]
df["Lower"] = df["MA"] - 2 * df["STD"]

# --- Define additional plots for mplfinance ---
add_plots = [
  mpf.make_addplot(df["MA"], color="blue",alpha=0.6),
  mpf.make_addplot(df["Upper"], color="green",alpha=0.4),
  mpf.make_addplot(df["Lower"], color="red",alpha=0.4),
]

# --- Plot ---
mpf.plot(
  df,
  type="candle",
  volume=True,
  style="charles",
  addplot=add_plots,
  title="JNK Candlesticks with Bollinger Bands & Volume",
  ylabel="Price USD",figsize=(14, 6), #ylim=(100, 370),
  ylabel_lower="Volume",xlabel="Date"
)

#### **AGG**

In [ ]:
lqd_df = fetch_eod("AGG", start_date, end_date)

# Visualization
df = lqd_df.copy()

# --- Bollinger Bands (20-period normally; here small dataset so just demo) ---
window = 20  # use 20 in real data
df["MA"] = df["close"].rolling(window).mean()
df["STD"] = df["close"].rolling(window).std()

df["Upper"] = df["MA"] + 2 * df["STD"]
df["Lower"] = df["MA"] - 2 * df["STD"]

# --- Define additional plots for mplfinance ---
add_plots = [
  mpf.make_addplot(df["MA"], color="blue",alpha=0.6),
  mpf.make_addplot(df["Upper"], color="green",alpha=0.4),
  mpf.make_addplot(df["Lower"], color="red",alpha=0.4),
]

# --- Plot ---
mpf.plot(
  df,
  type="candle",
  volume=True,
  style="charles",
  addplot=add_plots,
  title="AGG Candlesticks with Bollinger Bands & Volume",
  ylabel="Price USD",figsize=(14, 6), #ylim=(100, 370),
  ylabel_lower="Volume",xlabel="Date"
)

#### **VXX**

In [ ]:
vix_df = fetch_eod("VXX", start_date, end_date)

# Visualization
df = pd.DataFrame()
df = vix_df.copy()

# --- Bollinger Bands (20-period normally; here small dataset so just demo) ---
window = 20  # use 20 in real data
df["MA"] = df["close"].rolling(window).mean()
df["STD"] = df["close"].rolling(window).std()

df["Upper"] = df["MA"] + 2 * df["STD"]
df["Lower"] = df["MA"] - 2 * df["STD"]

# --- Define additional plots for mplfinance ---
add_plots = [
  mpf.make_addplot(df["MA"], color="blue",alpha=0.6),
  mpf.make_addplot(df["Upper"], color="green",alpha=0.4),
  mpf.make_addplot(df["Lower"], color="red",alpha=0.4),
]

# --- Plot ---
mpf.plot(
  df,
  type="candle",
  volume=True,
  style="charles",
  addplot=add_plots,
  title="VXX Candlesticks with Bollinger Bands & Volume",
  ylabel="Price USD",figsize=(14, 6), #ylim=(100, 370),
  ylabel_lower="Volume",xlabel="Date"
)

#### **Credit Risk Ratio**

In [ ]:
# --- Ratio ---
ratio = hyg_df["close"] / lqd_df["close"]


# --- Bollinger Bands ---
window = 20

ma = ratio.rolling(window).mean()
std = ratio.rolling(window).std()

upper = ma + 2 * std
lower = ma - 2 * std

# --- Plot ---
fig, ax = plt.subplots(figsize=(14, 6))

# Ratio line
ax.plot(ratio.index, ratio, label="JNK / AGG Ratio")

# Bollinger Bands
ax.plot(ma.index, ma, label="20D MA", alpha=0.9)
ax.plot(upper.index, upper, alpha=0.5)
ax.plot(lower.index, lower, alpha=0.5)

# Optional shaded band region
ax.fill_between(
  ratio.index,
  lower,
  upper,
  alpha=0.15
)

# Labels
ax.set_title("JNK / AGG Credit Risk Ratio with Bollinger Bands")
ax.set_ylabel("Ratio")
ax.set_xlabel("Date")
ax.grid(True)
ax.legend()

plt.show()

#### **Data Preparation**

In [ ]:
df = spy_df.copy()

df = df.sort_index()

df = df.join(
  hyg_df["adjusted_close"].rename("jnk"),
  how="inner"
)

df = df.join(
  lqd_df["adjusted_close"].rename("agg"),
  how="inner"
)

df["Credit_Spread"] = (
  np.log(df["jnk"]) - np.log(df["agg"])
).diff()

df = df.dropna()

win_s = 15 #short-term
win_m = 50 #medium-term
win_l = 150 #long-term

# Returns
df["SPX_Daily_Return"] = spy_df['close'].pct_change()
df["SPX_21D_Return"] = spy_df['close'].pct_change(win_s)
df["SPX_63D_Return"] = spy_df['close'].pct_change(win_m)
df["SPX_126D_Return"] = spy_df['close'].pct_change(win_l)

df["RUT_Daily_Return"] = iwm_df['close'].pct_change()
df["RUT_21D_Return"] = iwm_df['close'].pct_change(win_s)
df["RUT_63D_Return"] = iwm_df['close'].pct_change(win_m)
df["RUT_126D_Return"] = iwm_df['close'].pct_change(win_l)

df["SPX_vs_RUT_21D"] = np.log(spy_df["close"] / iwm_df['close']).diff(win_s)
df["SPX_vs_RUT_63D"] = np.log(spy_df["close"] / iwm_df['close']).diff(win_m)

In [ ]:
def realized_vol(series, span=21):
  series = series.copy() #Avoids modifying the original data
  series = series.replace([0, np.inf, -np.inf], np.nan) #Removes invalid values (0, infinity, -infinity)
  series = series.dropna() #Drops missing values to ensure clean computations
  returns = np.log(series).diff() #converts prices into continuous returns
  return returns.ewm(span=span).std() * np.sqrt(252) # computes exponentially weighted volatility

In [ ]:
df["SPX_21D_RealVol"] = realized_vol(df["SPX_Daily_Return"], span=win_s)
df["SPX_63D_RealVol"] = realized_vol(df["SPX_Daily_Return"], span=win_m)
df["RUT_21D_RealVol"] = realized_vol(df["RUT_Daily_Return"], span=win_s)
df["RUT_63D_RealVol"] = realized_vol(df["RUT_Daily_Return"], span=win_m)

In [ ]:
# Step 1: align VXX into df vix_df
df['vxx_close'] = vix_df['close']

# Step 2: clean missing values
df['vxx_close'] = df['vxx_close'].ffill().bfill()

# Step 3: rolling regime smoothing
df["vxx3m_rolling_avg"] = df['vxx_close'].rolling(window=win_m, min_periods=1).mean()
df["vxx6m_rolling_avg"] = df['vxx_close'].rolling(window=win_l, min_periods=1).mean()

df["VXX_1D_Change"] = vix_df["close"].diff(1)
df["VXX_5D_Change"] = vix_df["close"].diff(5)

df["VXX_to_SPXRealVol"] = vix_df["close"] / df["SPX_21D_RealVol"]
df["VXX3M_VXX"] = df["vxx3m_rolling_avg"] / vix_df["close"]
df["VXX6M_VXX"] = df["vxx6m_rolling_avg"] / vix_df["close"]

In [ ]:
# Drawdowns
spx_roll_max = spy_df["close"].rolling(win_l, min_periods=1).max()
rut_roll_max = iwm_df["close"].rolling(win_l, min_periods=1).max()
df["SPX_126D_Drawdown"] = spy_df["close"] / spx_roll_max - 1.0
df["RUT_126D_Drawdown"] = iwm_df["close"] / rut_roll_max - 1.0

In [ ]:
df = df.dropna().reset_index(drop=True)
display(df.tail())

In [ ]:
df["spx_close"] = spy_df['close']
df["rut_close"] = iwm_df['close']

columns_list = df.columns.tolist()
print(columns_list)

In [ ]:
exclude_exact = {
  "open", "high", "low", "close", "adjusted_close", "volume",  # raw OHLCV
  "spx_close", "rut_close",  # duplicate raw series
  "jnk", "agg",              # raw ETF prices (we already use spreads)
}

# Add any remaining helper/raw-like columns explicitly excluded
exclude_exact.update({
  # optional: keep or remove depending on modeling choice
  "vxx_close"
})

# Select numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Final feature set
feature_cols = [c for c in numeric_cols if c not in exclude_exact]

#Feature Selection
X_df = df[feature_cols].replace([np.inf, -np.inf], np.nan).dropna()
valid_idx = X_df.index

#cleaning

X_clean = X_df.replace([np.inf, -np.inf], np.nan)
X_clean = X_clean.dropna()

#Scaling
X_scaled = (X_clean - X_clean.expanding().mean()) / X_clean.expanding().std()
X_scaled = X_scaled.dropna()
valid_idx = X_scaled.index

#### **Cluster Analysis & Regime Visualization**

In [ ]:
# PCA reduction (variance target)
variance_target = 0.95
pca = PCA()
X_pca = pca.fit_transform(X_scaled)
explained_var = pca.explained_variance_ratio_

plt.figure(figsize=(8,5))
plt.plot(np.cumsum(explained_var), marker="o")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.title("PCA - Cumulative Variance Explained")
plt.grid(True)
plt.show()

#### **PCA & K-Means Clustering**

In [ ]:
n_components = np.argmax(np.cumsum(explained_var) >= variance_target) + 1
X_pca_reduced = X_pca[:, :n_components]
print(f"Selected {n_components} PCs (≈{np.cumsum(explained_var)[n_components-1]:.3f} variance)")

# Silhouette on RAW (scaled) vs PCA (reduced)
def silhouette_over_k(X, k_min=2, k_max=6, seed=42):
  scores = {}
  for k in range(k_min, k_max+1):
    km = KMeans(n_clusters=k, n_init=50, random_state=seed)
    labels = km.fit_predict(X)
    scores[k] = silhouette_score(X, labels)
  return scores

scores_raw = silhouette_over_k(X_scaled, 2, 6)
scores_pca = silhouette_over_k(X_pca_reduced, 2, 6)

print("\nSilhouette (raw scaled):", {k: round(v,3) for k,v in scores_raw.items()})
print("Silhouette (PCA reduced):", {k: round(v,3) for k,v in scores_pca.items()})

# Choose representation with higher best silhouette
best_k_raw = max(scores_raw, key=scores_raw.get)
best_k_pca = max(scores_pca, key=scores_pca.get)

use_pca = scores_pca[best_k_pca] >= scores_raw[best_k_raw]
rep = "PCA" if use_pca else "RAW"
best_k = best_k_pca if use_pca else best_k_raw
X_final = X_pca_reduced if use_pca else X_scaled

print(f"\nChose {rep} features with k={best_k} "
    f"(silhouette={max(scores_pca.values()) if use_pca else max(scores_raw.values()):.3f})")

# Fit final KMeans on chosen representation
kmeans = KMeans(n_clusters=best_k, n_init=50, random_state=42)
final_labels = kmeans.fit_predict(X_final)

# Write labels back to original df (only rows used after cleaning)
df["Regime"] = np.nan
df.loc[valid_idx, "Regime"] = final_labels
df["Regime"] = df["Regime"].astype("Int64")  # nullable int

print("\nRegime distribution (on modeled rows):")
print(pd.Series(final_labels).value_counts().sort_index())

# PCA loadings (interpretation)
loadings = pd.DataFrame(
  pca.components_.T,
  index=feature_cols,
  columns=[f"PC{i+1}" for i in range(len(pca.components_))]
)

def top_loadings(load_df, pc="PC1", n=8):
  s = load_df[pc].sort_values()
  return pd.concat([s.head(n), s.tail(n)])

print("\nTop loadings for PC1:")
display(top_loadings(loadings, "PC1"))
print("\nTop loadings for PC2:")
display(top_loadings(loadings, "PC2"))

#### **Plotting Regime Clusters in PC1 vs PC2 space (k=2, PCA)**

In [ ]:
# Use first 2 PCs for plotting
X_plot = X_final[:, :2]

# Prepare DataFrame
plot_df = pd.DataFrame(X_plot, columns=["PC1", "PC2"])
plot_df["Regime"] = final_labels.astype(int)

# Dynamic color palette
colors = ["green", "red", "blue", "orange", "purple", "brown", "pink"]

# Create mapping automatically
unique_regimes = sorted(plot_df["Regime"].unique())
color_map = {
  regime: colors[i % len(colors)]
  for i, regime in enumerate(unique_regimes)
}

# Plot
plt.figure(figsize=(10,7))

for regime in unique_regimes:
  subset = plot_df[plot_df["Regime"] == regime]

  plt.scatter(
    subset["PC1"],
    subset["PC2"],
    c=color_map[regime],
    label=f"Regime {regime}",
    s=60,
    alpha=0.8,
    edgecolor="k"
  )

plt.title(f"Regime Clusters in PC1 vs PC2 space (k={best_k}, {rep})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Regime")
plt.grid(True, linestyle="--", alpha=0.5)

plt.show()

In [ ]:
# Copy dataframe
df_plot = df.copy()

# Keep valid rows only
df_plot = df_plot.dropna(subset=["close", "Regime"]).copy()

# Convert Regime to int
df_plot["Regime"] = df_plot["Regime"].astype(int)

# Define colors for regimes
colors = ["green", "red", "blue", "orange", "purple"]

unique_regimes = sorted(df_plot["Regime"].unique())

color_map = {
  regime: colors[i % len(colors)]
  for i, regime in enumerate(unique_regimes)
}

# Create scatter plot
plt.figure(figsize=(14,6))

for regime in unique_regimes:
  subset = df_plot[df_plot["Regime"] == regime]

  plt.scatter(
    subset.index,
    subset["close"],
    color=color_map[regime],
    label=f"Regime {regime}",
    s=20,
    alpha=0.8
  )

plt.title("VTI Price Scatter Plot Colored by Regime", fontsize=14)
plt.xlabel("Time Index")
plt.ylabel("Close")

plt.legend(title="Regimes")

plt.grid(True, linestyle="--", alpha=0.5)

plt.show()

In [ ]:
df.groupby("Regime")["SPX_Daily_Return"].agg([
  "mean",
  "std",
  "min",
  "max"
])

In [ ]:
df.groupby("Regime")[[
  "SPX_21D_RealVol",
  "Credit_Spread",
  "VXX_1D_Change"
]].mean()


#### **Market Regime Classification & CV Pipeline**

In [ ]:
# Features and target
X = X_final               # our PCA-reduced features
y = df.loc[df["Regime"].notna(), "Regime"].astype(int).values  # target 0/1

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
  X, y, test_size=0.4, random_state=42, stratify=y
)

# Classifier

clf = LogisticRegression(max_iter=500, multi_class='auto', solver='lbfgs')
clf.fit(X_train, y_train)

# Predictions and accuracy
y_pred = clf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

# Generate classification report
report = classification_report(y_test, y_pred, labels=[0, 1], target_names=["Regime 0", "Regime 1"])
print("Classification Report:\n")
print(report)

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

# Option 1: using sklearn's built-in plot
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
disp.plot(cmap=plt.cm.Blues)
plt.title("Confusion Matrix: Regime 0 vs 1")
plt.show()

#### **Interpretative Machine Learning Dashboard**

##### *Normalized Confusion Matrix & Classification Report*

In [ ]:
# --- Compute confusion matrix ---
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100  # % per actual class

# --- Get classification report as DataFrame ---
report_dict = classification_report(
    y_test, y_pred, labels=[0,1], target_names=["Regime 0", "Regime 1"], output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose().round(2)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14,5))

# Confusion matrix heatmap
sns.heatmap(
    cm_normalized,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    xticklabels=[0,1],
    yticklabels=[0,1],
    ax=axes[0]
)
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
axes[0].set_title("Normalized Confusion Matrix (% per actual class)")

# Classification report table
axes[1].axis('off')  # hide axes
axes[1].table(
    cellText=report_df.values,
    colLabels=report_df.columns,
    rowLabels=report_df.index,
    cellLoc="center",
    loc="center"
)
axes[1].set_title("Classification Report")

plt.tight_layout()
plt.show()

##### *Learning Curves*

In [ ]:
skplt.estimators.plot_learning_curve(LogisticRegression(), X, y,
                                     cv=7, shuffle=True, scoring="accuracy",
                                     n_jobs=-1, figsize=(12,6), title_fontsize="large", text_fontsize="large",
                                     title="LR Regime Classification Learning Curve");
plt.grid()
plt.show()

##### *Calibration Plots*

In [ ]:
lr_probas = LogisticRegression().fit(X_train, y_train).predict_proba(X_test)
rf_probas = RandomForestClassifier().fit(X_train, y_train).predict_proba(X_test)
gb_probas = GradientBoostingClassifier().fit(X_train, y_train).predict_proba(X_test)
et_scores = ExtraTreesClassifier().fit(X_train, y_train).predict_proba(X_test)

probas_list = [lr_probas, rf_probas, gb_probas, et_scores]
clf_names = ['Logistic Regression', 'Random Forest', 'Gradient Boosting', 'Extra Trees Classifier']

skplt.metrics.plot_calibration_curve(y_test, probas_list, clf_names, n_bins=15, figsize=(12,6))

##### *KS Statistic Plot*

In [ ]:
rf = log_reg
rf.fit(X_train, y_train)
y_probas = rf.predict_proba(X_test)

skplt.metrics.plot_ks_statistic(y_test, y_probas, figsize=(10,6))

##### *Yellowbrick ML Visualization*

In [ ]:
from yellowbrick.classifier import ClassificationReport

viz = ClassificationReport(LogisticRegression(random_state=123), classes=target_names,
                           support=True, fig=plt.figure(figsize=(10, 8)))

viz.fit(X_train, y_train)
viz.score(X_test, y_test)
viz.show()

##### *ROC-AUC Curve*

In [ ]:
from yellowbrick.classifier import ROCAUC

viz = ROCAUC(LogisticRegression(random_state=123),
             classes=target_names,
             fig=plt.figure(figsize=(7,5)))

viz.fit(X_train, y_train)

viz.score(X_test, y_test)

viz.show();

##### *Precision-Recall (PR) Curve*

In [ ]:
from yellowbrick.classifier import PrecisionRecallCurve

viz = PrecisionRecallCurve(LogisticRegression(random_state=123),
             classes=target_names, ap_score=True,
             iso_f1_curves=True, fig=plt.figure(figsize=(10, 8)))

viz.fit(X_train, y_train)
viz.score(X_test, y_test)
viz.show()

##### *Discrimination Threshold*

In [ ]:
from yellowbrick.classifier import DiscriminationThreshold

viz = DiscriminationThreshold(LogisticRegression(random_state=123),
             classes=target_names, cv=0.2,
             fig=plt.figure(figsize=(10, 8)))

viz.fit(X_train, y_train)
viz.score(X_test, y_test)
viz.show()

In [ ]:
from yellowbrick.classifier import ClassPredictionError

viz = ClassPredictionError(LogisticRegression(random_state=123),
                           classes=target_names, fig=plt.figure(figsize=(9,6)))

viz.fit(X_train, y_train)
viz.score(X_test, y_test)
viz.show()